Hash Implementation

Before diving into this chapter, we would like to preface it by saying that it is unlikely that you are asked to code the implementation of a hash map from scratch in an interview. However, the knowledge we will cover here will be useful, not only because it widely applies to many algorithms, but also because it applies to systems design & distributed systems.

Hash maps are most commonly implemented using arrays under the hood. While an empty hash map is of size zero, initially our array will not be of size zero.

Suppose that we want to fill up our array with the follow key-value pairs.

To do so in O(1) time, we need to introduce a hash function. A hash function takes the key (a string in this case) and converts it into an integer. This integer will be the index at which we store our key-value pair in the array.

Of course we need to ensure that it is a valid index and also that two different keys aren't mapped to the same index.

The same string will always result in the same integer. Using this integer, we can determine the location at which we wish to store our key-value pair.


Insertion and Hashing

Consider the string "Alice" as a key.

1. To convert it into an integer, our hash function will take each character in the string and get its ASCII code.
2. Then it will add up the ASCII codes to determine where it should end up in the array.
3. However, since this number may be large and out of bounds of the array, we can use the modulo operator to find a valid index.

Let's imagine that summing up all the ASCII codes for all the characters in "Alice" is 25. To determine its location in the array, we can use the formula: sum(ASCII codes) % size(array).

In this case, the size of the array space is 2. So, 25 % 2 results in 1, which is the index that we store the first key value pair. The process is also sometimes known as pre-hashing.

Because the size of our array is only 2, it is possible that another key-value pair results in the same array position when we take the mod. This is called a collision and collisions are very common. There are ways to try to minimize collisions but they are inevitable.

In [ ]:
hashmap["Alice"] = "NYC"
hashmap["Brad"] = "Chicago"
hashmap["Collin"] = "Seattle"

Resizing and Re-hashing


Resizing
To ensure each key-value pair finds a vacant spot, we will keep track of the size of the array, and the number of positions that are actually non-empty. At some point we will run out of space and need to resize the array.

We can create a new array with a capacity of 2 * capacity, where capacity is the size of the current array. To try to minimize collisions we will do this when the array is half full instead of completely full.

In this example, we have a size of 2 and because adding "Alice" : "NYC" will result in the map being half full, we will double its size before performing the next insertion. The resizing works the same as we described in the dynamic arrays chapter.

We don't resize the array at the time of insertion of a new key-value pair, but rather as soon as the array becomes half full. This ensures that we minimize collisons before insertion.


Rehashing
Once we perform the resize, we perform an operation called re-hashing. We need to reinsert all of the key-value pairs into the new array, but this is not as straightforward as it seems.

Since the size of the array has changed, the position of the key-value pairs will also change. We need to recompute the position of all the elements that already exist in the hash map using the same formula, sum(ASCII codes) % size(array), but with a different capacity.

This is needed because the size of the array has changed. Rehashing tells us to re-compute the position of all the elements that already exist in the hash map. We inserted "Alice" at index 1. After doubling the size of the array, the new position of "Alice", in this case, just happens to be at index 1 again. However, if "Alice"'s new position was now at index 2, we would place it there instead.

Suppose that converting "Brad" to integer results in 27. 27 mod 4 = 3, meaning that "Brad" : "Chicago" would end up in the 3rd index. Now, we will double the size to be size 8.


Collisions
Suppose converting "Collin" to an integer results in 33. 33 mod 8 = 1. This is a collision since Alice already resides in the first index. How do we go about resolving this?

- We could just overwrite the previous key, but that would mean that we end up losing "Alice".
- We could also keep increasing the size of the array until we find a vacant space for "Collin". This, may not necessarily resolve the issue, and would waste a lot of memory.

There are two common ways in which we can deal with collisions:

1. Chaining
Chaining (recall the concept from the Linked List chapter) is achieved by chaining Linked List nodes together, so that multiple key-value pairs can be stored at the same index.

Because Alice and Collin belong to the same index, we can store them as a linked list object. In this case, the time complexity for searching and inserting could become O(n) if all of the key-value pairs end up in the same index. However, the average time complexity is still O(1).

2. Open Addressing

The idea behind open addressing is to find the next available slot so that we don't store more than one key-value pair in one index. For example, if we want to insert at index 1 but it is already occupied, we can try inserting at index 2, and 3 and so on until we find an empty slot.

When we lookup a key, we will also need to check the next available slots until we find the key we are looking for, and then we can return the cooresponding value.

In practice, this is much more complicated to implement than chaining, so if you do not have a specific reason to use open addressing, it is simpler to stick with chaining.

This technique, compared to chaining is more efficient if there are a small number of collisions. The limitation here however is that the total number of entries in the table is limited by the size of the array.

To reduce the amount of collisions that occur, it mathematically makes sense to choose the hashmap to be of a prime size. This is because the prime number is only divisible by 1 and itself! This post from CS StackExchange provides a mathematical proof if you are interested.

Code Implementation

Now that we understand the concept behind hashing, its implementation, searching, deletion, insertion and how a hash function works, let's take a look at how we would go about implementing it in code with open addressing.

In [ ]:
# We will store a list of Pairs in our array, for which we declare a class.

class Pair:
    def __init__(self, key, val):
        self.key = key
        self.val = val

In [ ]:
# We initialize a size, capacity and the map itself in our constructor. 
# Here, size refers to the size of the hash map and capacity 
# refers to size of the array under the hood.

class HashMap:
    def __init__(self):
        self.size = 0
        self.capacity = 2
        self.map = [None, None]

In [ ]:
# The hash function below iterates through each of the characters in a given key, 
# sums up their ASCII code and finds the position of the key in the array.

def hash(self, key):
    index = 0
    for c in key:
        index += ord(c)
    return index % self.capacity

In [ ]:
# To retrieve the value, we first need to retrieve the position and check if the value exists 
# in that position. If it does, we can return that value. Otherwise, 
# we can perform open addressing and look for it in the next available index.

def get(self, key):
    index = self.hash(key)
    
    while self.map[index] != None:
        if self.map[index].key == key:
            return self.map[index].val
        index += 1
        index = index % self.capacity
    return None

In [ ]:
# To add to the map, we first compute the hash of the key and find the position. 
# Once this is calculated, there are three scenarios.
    # 1. The index is occupied
    # 2. The index is occupied with the same key
    # 3. The index is vacant

def put(self, key, val):
    index = self.hash(key)

    while True:
        if self.map[index] == None:
            self.map[index] = Pair(key, val)
            self.size += 1
            if self.size >= self.capacity // 2:
                self.rehash()
            return
        elif self.map[index].key == key:
            self.map[index].val = val
            return
        
        index += 1
        index = index % self.capacity

In [ ]:
# To remove, we find the index, remove the key, and set the index to null.

def remove(self, key):
    if not self.get(key):
        return
    
    index = self.hash(key)
    while True:
        if self.map[index].key == key:
            # Removing an element using open-addressing actually causes a bug,
            # because we may create a hole in the list, and our get() may 
            # stop searching early when it reaches this hole.
            self.map[index] = None
            self.size -= 1
            return
        index += 1
        index = index % self.capacity

In [ ]:
# When we perform re-hashing, we double the capacity, copy our previous map's values 
# into our new map and set the size to be zero.

def rehash(self):
    self.capacity = 2 * self.capacity
    newMap = []
    for i in range(self.capacity):
        newMap.append(None)

    oldMap = self.map
    self.map = newMap
    self.size = 0
    for pair in oldMap:
        if pair:
            self.put(pair.key, pair.val)

In [ ]:
# Finally, if we wish to print all of the pairs, it would look like the following.

def print(self):
    for pair in self.map:
        if pair:
            print(pair.key, pair.val)

Time Complexity: 

Operation  | HashMap
Insert     | O(1)   
Remove     | O(1)     
Search     | O(1)    

The time complexity for all operations is O(1) on average, but this is only the case if we have a good hash function and a low number of collisions. In the worst case, the time complexity could be O(n).